# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use **Random Forest** for this modeling lane.

Random Forest fits this dataset because the available signals are numeric and highly skewed, and the relationships between impressions, clicks, position, and traffic may not be purely linear. It can capture non-linear patterns while remaining relatively easy to inspect with feature importance.

I will use the model as **decision-support**, not as a causal explanation. The model will be evaluated on held-out clients. Because the Week-4 baseline is a rule-based prioritization score without a supervised evaluation metric, I will not claim a direct metric comparison where one is not valid.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
import pandas as pd

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Rows:", len(df_march))
print("Columns:", df_march.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Split design

I will use a **client-grouped split**.

The same client can have many daily observations, so I will keep each client entirely in either the training set or the test set. This reduces the risk of learning client-specific patterns and gives a more honest check of whether the model can generalize to unseen clients.

I will use an 80/20 client-level split and keep the test set untouched until evaluation.
The Week-4 baseline is a prioritization score rather than a supervised prediction model, so the comparison will focus on whether the model provides additional predictive value without treating the baseline score as a ground-truth label.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

model_df = df_march.copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Total rows:", len(model_df))
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Client overlap:", len(overlap))

Total rows: 9841378
Train rows: 8935676
Test rows: 905702
Train clients: 44
Test clients: 11
Client overlap: 0


## 3. Train + compare vs my baseline

The Week-4 baseline is a rule-based prioritization score, not a supervised prediction model. Therefore, I will not treat the baseline action as a ground-truth label.

For this modeling step, I will predict whether a page receives at least one GSC click using signals available in the same March 2026 dataset. I will report the model metric separately and use the Week-4 score as a decision-support baseline rather than claiming a direct apples-to-apples performance comparison.

This is a same-row classification exercise: the target indicates whether clicks were observed for that daily record. It should not be interpreted as a forecast of future clicks because the current dataset does not provide a future outcome window. The held-out client split is used to test generalization across clients.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_ai"
]

target = "gsc_clicks"

train_model = train_df[features + [target]].copy()
test_model = test_df[features + [target]].copy()

# Binary target: 1 = at least one click, 0 = no clicks
train_model["clicked"] = (train_model[target] > 0).astype(int)
test_model["clicked"] = (test_model[target] > 0).astype(int)

# Replace missing numeric values with 0 for this simple baseline model
X_train = train_model[features].fillna(0)
X_test = test_model[features].fillna(0)

y_train = train_model["clicked"]
y_test = test_model["clicked"]

print("Features:", features)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

Features: ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'sessions_ai']
Train rows: 8935676
Test rows: 905702
Train positive rate: 0.0427484165719527
Test positive rate: 0.03974265266058814


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

model.fit(X_train, y_train)

test_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, test_prob)
pr_auc = average_precision_score(y_test, test_prob)

print("Random Forest ROC-AUC:", round(roc_auc, 4))
print("Random Forest PR-AUC:", round(pr_auc, 4))

Random Forest ROC-AUC: 0.9769
Random Forest PR-AUC: 0.7404


In [6]:
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("Feature importance:")
display(importance.to_frame("importance"))

Feature importance:


,importance
gsc_impressions,0.642549
gsc_avg_position,0.281966
ga4_sessions,0.075394
sessions_ai,0.000091


In [14]:
print("MODEL VS WEEK-4 BASELINE")
print("-" * 60)

print("Week-4 baseline")
print("  Type: Rule-based prioritization")
print("  Metric: Not defined in Week-4")
print("  Result: Not directly comparable")

print("\nRandom Forest")
print("  Type: Supervised classifier")
print("  Metric: ROC-AUC / PR-AUC")
print(f"  Result: ROC-AUC={roc_auc:.4f}; PR-AUC={pr_auc:.4f}")

MODEL VS WEEK-4 BASELINE
------------------------------------------------------------
Week-4 baseline
  Type: Rule-based prioritization
  Metric: Not defined in Week-4
  Result: Not directly comparable

Random Forest
  Type: Supervised classifier
  Metric: ROC-AUC / PR-AUC
  Result: ROC-AUC=0.9769; PR-AUC=0.7404


## 4. Errors and interpretation

On the held-out client set, the Random Forest achieved a ROC-AUC of 0.9769 and a PR-AUC of 0.7404. At the 0.5 classification threshold, the measured false positive rate was 12.52% and the false negative rate was 4.62%.

The feature importance results show that the model leans most on gsc_impressions (0.6425) and gsc_avg_position (0.2820). ga4_sessions contributes less (0.0754), while sessions_ai has very little importance (0.0001).

These results are directional rather than causal. Some errors are expected because a page can receive impressions without clicks for reasons not represented in this dataset, such as search intent or SERP context. The model should therefore be treated as decision-support rather than as a definitive explanation of why a page receives clicks.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix


test_pred = (test_prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred
).ravel()

print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

print("\nFalse positive rate:", round(fp / (fp + tn), 4))
print("False negative rate:", round(fn / (fn + tp), 4))

True negatives: 760779
False positives: 108928
False negatives: 1663
True positives: 34332

False positive rate: 0.1252
False negative rate: 0.0462


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.